# Machines Don't Understand Text
Computers and machine learning algorithms work with numbers, not words. When you have text data like:

- "I love this movie"
- "This movie is terrible"
- "Great film, highly recommend"

**The challenge:** How do we convert these text strings into numbers that algorithms can process while preserving meaningful information?

# What is Bag of Words?

Bag of Words (BoW) is a fundamental text representation technique that converts text documents into numerical vectors that machine learning algorithms can understand and process. It treats each document as an unordered collection (or "bag") of words, ignoring grammar, word order, and sentence structure while maintaining word frequency information.

In [4]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [5]:
texts = [
    'Ini adalah laptop, tapi itu adalah handphone',
    'Itu adalah laptop.',
    'Saya mau beli handphone.',
    'Saya ada handphone itu, tapi tidak ada laptop itu.',
    'Saya tidak ada laptop ini',
    'Saya mau beli handphone dan laptop',
    'Ini adalah handphone',
    'Saya mau beli handphone, which is gak beli laptop'
]

In [11]:
bow = CountVectorizer()
bow_matrix = bow.fit_transform(texts)

<8x15 sparse matrix of type '<class 'numpy.int64'>'
	with 42 stored elements in Compressed Sparse Row format>

In [14]:
pd.DataFrame(bow_matrix.toarray(), columns=bow.get_feature_names_out(), index=texts)

,ada,adalah,beli,dan,gak,handphone,ini,is,itu,laptop,mau,saya,tapi,tidak,which
"Ini adalah laptop, tapi itu adalah handphone",0,2,0,0,0,1,1,0,1,1,0,0,1,0,0
Itu adalah laptop.,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0
Saya mau beli handphone.,0,0,1,0,0,1,0,0,0,0,1,1,0,0,0
"Saya ada handphone itu, tapi tidak ada laptop itu.",2,0,0,0,0,1,0,0,2,1,0,1,1,1,0
Saya tidak ada laptop ini,1,0,0,0,0,0,1,0,0,1,0,1,0,1,0
Saya mau beli handphone dan laptop,0,0,1,1,0,1,0,0,0,1,1,1,0,0,0
Ini adalah handphone,0,1,0,0,0,1,1,0,0,0,0,0,0,0,0
"Saya mau beli handphone, which is gak beli laptop",0,0,2,0,1,1,0,1,0,1,1,1,0,0,1


# The Problem with Basic Word Counting

Imagine you have these documents:
- Document 1: "I love this movie"
- Document 2: "This movie is great" 
- Document 3: "I think this is good"
  
Using basic **Bag of Words**, the word "this" appears in all 3 documents, so it gets high importance. But "this" doesn't really tell us much about what makes each document unique!

**The Problem:** Common words like "this", "is", "the" appear everywhere but don't carry meaningful information.

# What is Inverse Document Frequency (IDF)?

**Inverse Document Frequency (IDF)** is a numerical measure used in text analysis and information retrieval to evaluate how important a word is across a collection of documents. 

**Purpose**: IDF reduces the importance of common words and boosts the importance of rare words.

## Why Do We Need IDF?

### The Problem with Simple Word Counting

When analyzing text documents, simply counting word frequencies can be misleading:

| Word Type | Example | Problem |
|-----------|---------|---------|
| Common Words | "the", "is", "and", "this" | Appear in almost every document but carry little meaning |
| Rare Words | "quantum", "blockchain", "photosynthesis" | Appear in few documents but are highly informative |

**Without IDF**: Common words dominate the analysis despite being uninformative.

**With IDF**: Rare, meaningful words get higher importance scores.\

## How IDF Works

### Simple Logic

The logic behind IDF is straightforward:

| Condition | Interpretation | IDF Score |
|-----------|----------------|-----------|
| Word appears in **many documents** | Probably not very informative | **Lower score** |
| Word appears in **few documents** | Probably more meaningful | **Higher score** |

## Step-by-Step Example Calculate IDF

### Sample Documents

Let's analyze these 3 documents:

| Document | Content |
|----------|---------|
| **Doc 1** | "I love this movie" |
| **Doc 2** | "This movie is great" |
| **Doc 3** | "I think this is good" |

### Step 1: Count Document Frequencies

| Word | Doc 1 | Doc 2 | Doc 3 | **Documents Containing Word** |
|------|-------|-------|-------|------------------------------|
| "this" | ✓ | ✓ | ✓ | **3** |
| "love" | ✓ | ✗ | ✗ | **1** |
| "movie" | ✓ | ✓ | ✗ | **2** |
| "is" | ✗ | ✓ | ✓ | **2** |
| "great" | ✗ | ✓ | ✗ | **1** |

### Step 2: Calculate IDF Scores

| Word | Formula | Calculation | **IDF Score** |
|------|---------|-------------|---------------|
| "this" | log(3/3) | log(1) | **0.00** ← Low importance |
| "love" | log(3/1) | log(3) | **1.10** ← High importance |
| "movie" | log(3/2) | log(1.5) | **0.41** ← Medium importance |
| "is" | log(3/2) | log(1.5) | **0.41** ← Medium importance |
| "great" | log(3/1) | log(3) | **1.10** ← High importance |

### Step 3: Interpretation

| IDF Score Range | Interpretation | Example Words |
|-----------------|----------------|---------------|
| **0.0 - 0.2** | Very common words | "this", "the", "and" |
| **0.2 - 0.6** | Moderately common words | "movie", "is" |
| **0.6 - 1.0+** | Rare, informative words | "love", "great" |

## IDF Score Interpretation

### Score Ranges and Meanings

| IDF Score | Word Rarity | Informational Value | Example Scenario |
|-----------|-------------|--------------------|--------------------|
| **0** | Appears in all documents | Very low | Stop words ("the", "and") |
| **0.1 - 0.5** | Appears in most documents | Low | Common words ("movie", "good") |
| **0.5 - 1.0** | Appears in some documents | Medium | Moderately specific terms |
| **1.0+** | Appears in few documents | High | Domain-specific terms |

### Real-World Example

In a collection of 1000 news articles:

| Word | Appears in Documents | IDF Score | Interpretation |
|------|---------------------|-----------|----------------|
| "the" | 999 documents | 0.001 | Meaningless |
| "news" | 500 documents | 0.301 | Somewhat informative |
| "cryptocurrency" | 10 documents | 2.000 | Highly informative |
| "blockchain" | 5 documents | 2.301 | Very informative |

# What is TF-IDF?

**Term Frequency-Inverse Document Frequency (TF-IDF)** is a numerical statistic that reflects how important a word is to a specific document within a collection of documents. It combines two key concepts:

- Term Frequency (TF): How often a word appears in a document
- Inverse Document Frequency (IDF): How rare or common a word is across all documents

**Purpose:** TF-IDF identifies words that are frequent in specific documents but rare across the entire collection - these are typically the most informative terms.

In [17]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(texts)

In [21]:
pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out(), index=texts)

,ada,adalah,beli,dan,gak,handphone,ini,is,itu,laptop,mau,saya,tapi,tidak,which
"Ini adalah laptop, tapi itu adalah handphone",0.000000,0.694300,0.000000,0.000000,0.000000,0.239873,0.347150,0.000000,0.347150,0.239873,0.000000,0.000000,0.402298,0.000000,0.000000
Itu adalah laptop.,0.000000,0.635327,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.635327,0.438998,0.000000,0.000000,0.000000,0.000000,0.000000
Saya mau beli handphone.,0.000000,0.000000,0.569823,0.000000,0.000000,0.393735,0.000000,0.000000,0.000000,0.000000,0.569823,0.442240,0.000000,0.000000,0.000000
"Saya ada handphone itu, tapi tidak ada laptop itu.",0.628134,0.000000,0.000000,0.000000,0.000000,0.187265,0.000000,0.000000,0.542028,0.187265,0.000000,0.210334,0.314067,0.314067,0.000000
Saya tidak ada laptop ini,0.530845,0.000000,0.000000,0.000000,0.000000,0.000000,0.458075,0.000000,0.000000,0.316520,0.000000,0.355513,0.000000,0.530845,0.000000
Saya mau beli handphone dan laptop,0.000000,0.000000,0.427598,0.591265,0.000000,0.295461,0.000000,0.000000,0.000000,0.295461,0.427598,0.331860,0.000000,0.000000,0.000000
Ini adalah handphone,0.000000,0.635327,0.000000,0.000000,0.000000,0.438998,0.635327,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
"Saya mau beli handphone, which is gak beli laptop",0.000000,0.000000,0.570422,0.000000,0.394377,0.197075,0.000000,0.394377,0.000000,0.197075,0.285211,0.221352,0.000000,0.000000,0.394377


## IDF can be used to check the importance of words and stop words can be removed.

In [23]:
df = pd.DataFrame(tfidf.get_feature_names_out(), columns=["vocab"])
df["idf"] = tfidf.idf_
df.sort_values("idf")

,vocab,idf
5,handphone,1.251314
9,laptop,1.251314
11,saya,1.405465
1,adalah,1.810930
2,beli,1.810930
6,ini,1.810930
8,itu,1.810930
10,mau,1.810930
0,ada,2.098612
12,tapi,2.098612


# N-grams

## What are N-grams?

**N-grams** are sequences of N consecutive words (or characters) from a text. They capture **word order** and **context** that simple word counting methods miss.

## Types of N-grams

| N-gram Type | N Value | Example from "I love this movie" |
|-------------|---------|-----------------------------------|
| **Unigram** | 1 | "I", "love", "this", "movie" |
| **Bigram** | 2 | "I love", "love this", "this movie" |
| **Trigram** | 3 | "I love this", "love this movie" |
| **4-gram** | 4 | "I love this movie" |

## Why Use N-grams?

### Problem with Simple Words (Unigrams)
```
Text: "not good"
Unigrams: ["not", "good"] 
Problem: Loses the negative meaning!
```

### Solution with Bigrams
```
Text: "not good"  
Bigrams: ["not good"]
Result: Preserves the negative context ✓
```

## Quick Example

**Text**: "Saya sangat suka film ini"

| N-gram Type | Output |
|-------------|--------|
| **Unigrams** | ["Saya", "sangat", "suka", "film", "ini"] |
| **Bigrams** | ["Saya sangat", "sangat suka", "suka film", "film ini"] |
| **Trigrams** | ["Saya sangat suka", "sangat suka film", "suka film ini"] |

## Common Applications

| Use Case | Best N-gram | Why |
|----------|-------------|-----|
| **Language Models** | Bigrams/Trigrams | Predict next word based on previous words |
| **Sentiment Analysis** | Bigrams | Capture phrases like "not good", "very bad" |
| **Text Classification** | Unigrams + Bigrams | Balance between context and vocabulary size |
| **Spell Checking** | Character N-grams | Find similar word patterns |

## Advantages vs Disadvantages

| ✅ Advantages | ❌ Disadvantages |
|---------------|------------------|
| Captures word order | Vocabulary size grows exponentially |
| Preserves context | More sparse data |
| Better than simple words | Computationally more expensive |
| Easy to understand | May overfit to training data |


## When to Use N-grams

| Use N-grams When | Don't Use When |
|------------------|----------------|
| ✅ Context matters ("not good") | ❌ Memory/computation is limited |
| ✅ Word order is important | ❌ Vocabulary is already very large |
| ✅ Working with phrases | ❌ Need semantic understanding |
| ✅ Building language models | ❌ Documents are very short |

## Key Takeaway

**N-grams help preserve context and word order in text analysis**, making them especially useful when the meaning depends on word combinations rather than individual words alone.

In [24]:
bow = CountVectorizer(ngram_range=(1, 2))
bow_matrix = bow.fit_transform(texts)

pd.DataFrame(bow_matrix.toarray(), columns=bow.get_feature_names_out(), index=texts)

,ada,ada handphone,ada laptop,adalah,adalah handphone,adalah laptop,beli,beli handphone,beli laptop,dan,...,saya ada,saya mau,saya tidak,tapi,tapi itu,tapi tidak,tidak,tidak ada,which,which is
"Ini adalah laptop, tapi itu adalah handphone",0,0,0,2,1,1,0,0,0,0,...,0,0,0,1,1,0,0,0,0,0
Itu adalah laptop.,0,0,0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Saya mau beli handphone.,0,0,0,0,0,0,1,1,0,0,...,0,1,0,0,0,0,0,0,0,0
"Saya ada handphone itu, tapi tidak ada laptop itu.",2,1,1,0,0,0,0,0,0,0,...,1,0,0,1,0,1,1,1,0,0
Saya tidak ada laptop ini,1,0,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,1,1,0,0
Saya mau beli handphone dan laptop,0,0,0,0,0,0,1,1,0,1,...,0,1,0,0,0,0,0,0,0,0
Ini adalah handphone,0,0,0,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
"Saya mau beli handphone, which is gak beli laptop",0,0,0,0,0,0,2,1,1,0,...,0,1,0,0,0,0,0,0,1,1
